<a href="https://colab.research.google.com/github/EthanIsaac/valorant-voiceline-qa/blob/main/notebooks/01_agent_voice_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Agent Voice Classifier: fine-tuning 3 audio models

**Project:** VALORANT Voice-Line QA & Localization Assistant (Riot Games)
**Pipeline being fine-tuned:** `audio-classification`, which identifies **which of the 26 agents** is speaking in a voice-line recording.

**Why it matters:** each agent ships with hundreds of voice lines, recorded and localized into many languages. Audio QA must confirm that every file contains the right agent's voice before it reaches the game.

**Notebook outline**
- Step 0: Setup (packages, seed, output folder)
- Step 1: Load the dataset at a pinned revision
- Step 2: Clean the data and split it into train / validation / test
- Step 3: Choose the 3 candidate pre-trained models and prepare the audio
- Step 4: Fine-tune and evaluate each model (accuracy and macro-F1)
- Save and download the results (with SHA-256 fingerprints)

All results are written to `results/agent_classifier/` so the report can quote them exactly.

> Runtime: **GPU** (Runtime → Change runtime type → T4 GPU).

# Step 0: Setup

In [ ]:
# Same transformers constraint as the course notebooks
# (pandas, scikit-learn, soundfile, librosa and huggingface_hub are already installed in Colab).
# Expected: a pip "dependency conflicts" message about diffusers/gradio needing a newer
# huggingface-hub. Those packages are not used here, so it can be ignored.
%pip install -q "transformers>=4.46.0,<5.0.0" accelerate

In [ ]:
import io
import json
import os
import re
import time

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import transformers
from IPython.display import Audio, display
from transformers import set_seed

# One seed for everything (Python, NumPy, PyTorch), so reruns give the same results
SEED = 42
set_seed(SEED)

# Every result file produced by this notebook goes in this folder
RESULTS_DIR = "results/agent_classifier"
os.makedirs(RESULTS_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers {transformers.__version__} | torch {torch.__version__} | device: {device}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Step 1: Load the dataset

**Source:** [NgThVinh/ValorantAgentVoiceLines](https://huggingface.co/datasets/NgThVinh/ValorantAgentVoiceLines), in-game voice lines of every VALORANT agent with their transcripts (collected from the VALORANT Fandom wiki).
Each agent is stored as a separate file. The **agent name is the label** we want to predict.

We load a **fixed revision** (commit hash) so the data can never change under us. The download is about 2 GB and takes a couple of minutes.

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

DATASET = "NgThVinh/ValorantAgentVoiceLines"
REVISION = "c2e0d3493f44da5be07090d6ebb9b69ebbcb2060"   # pinned commit

# One parquet file per agent: "<agent>/train-00000-of-00001.parquet"
parquet_files = sorted(
    f for f in list_repo_files(DATASET, repo_type="dataset", revision=REVISION) if f.endswith(".parquet")
)

frames = []
for file in parquet_files:
    path = hf_hub_download(DATASET, file, repo_type="dataset", revision=REVISION)
    agent_df = pd.read_parquet(path)
    agent_df["agent"] = file.split("/")[0]
    frames.append(agent_df)

data = pd.concat(frames, ignore_index=True)
data["audio_bytes"] = data["audio_file"].map(lambda audio: audio["bytes"])   # raw WAV file content
data = data.drop(columns="audio_file")

AGENTS = sorted(data["agent"].unique())
print(f"{len(data)} voice lines from {len(AGENTS)} agents")
print(data.groupby("agent").size().to_string())

In [ ]:
# Listen to a few examples
for _, row in data.sample(3, random_state=SEED).iterrows():
    print(f"{row['agent']:<10} {row['audio_name']:<28} {row['transcript']}")
    display(Audio(row["audio_bytes"]))

# Step 2: Clean the data and split it

**Audio checks.** Every clip is a mono WAV recorded at 48 kHz. Clips shorter than **0.5 s** (e.g. "Mm-hmm") carry too little voice to identify a speaker, so they are removed.

**Transcript cleaning.** The transcripts contain wiki markup. The words actually spoken are always the parts **inside double quotes**:

| Raw transcript | Spoken text |
|---|---|
| `"Hey hermanos, spend everything you have." (''Hermanos'': Siblings)` | `Hey hermanos, spend everything you have.` (translator note removed) |
| `''sighs'' "Cool"` | `Cool` (stage direction removed) |
| `"Sure, but if anything explodes again, you are telling Brimstone\|Brim."` | `... you are telling Brim.` (wiki link: the second name is the one spoken) |
| `''Jett groans''` | *(no speech)* |

Two flags are added for the speech-to-text notebook: `has_speech` (the clip contains words) and `english_only` (no foreign words or translator notes).
Non-speech clips (groans, sighs) are **kept** for agent identification, because they are still the agent's voice.

**Train / validation / test split (80 / 10 / 10).** Many lines were recorded several times (e.g. two takes of *"Keep up."* by the same agent).
If one take were in training and another in test, the test would be too easy (**leakage**). So the split is made **per agent and by line**:
all takes of the same line by the same agent go to the same split. Every agent appears in all three splits.

In [ ]:
MIN_DURATION_S = 0.5
SPLIT_FRACTIONS = {"train": 0.8, "validation": 0.1, "test": 0.1}


def spoken_text(transcript):
    """Keep only the quoted (spoken) parts; in wiki links 'Name|Nickname' keep the spoken nickname."""
    quoted = re.findall(r'"([^"]*)"', str(transcript))
    text = " ".join(part.strip() for part in quoted)
    text = re.sub(r"[^\s|]+\|([^\s,.!?]+)", r"\1", text)
    return " ".join(text.split())


data["duration_s"] = data["audio_bytes"].map(lambda b: sf.info(io.BytesIO(b)).duration)
data["spoken_text"] = data["transcript"].map(spoken_text)
data["has_speech"] = data["spoken_text"] != ""
data["english_only"] = (
    data["has_speech"]
    & ~data["transcript"].str.contains(r"\(", regex=True)                       # no translator note
    & ~data["spoken_text"].str.contains(r"[^\x00-\x7F’‘“”…—–]", regex=True)   # no non-English characters
)

cleaning_log = [("original", len(data))]
data = data[data["duration_s"] >= MIN_DURATION_S].reset_index(drop=True)
cleaning_log.append((f"removed clips shorter than {MIN_DURATION_S}s", len(data)))

print(pd.DataFrame(cleaning_log, columns=["step", "clips"]).to_string(index=False))
print(f"\nDuration (s): median {data['duration_s'].median():.2f}, 95th pct {data['duration_s'].quantile(0.95):.2f}, max {data['duration_s'].max():.2f}")
print(f"Clips with speech: {data['has_speech'].sum()} | English-only: {data['english_only'].sum()}")

# Before/after examples of the transcript cleaning
data.loc[data["transcript"].str.contains(r"\(|''|\|", regex=True), ["transcript", "spoken_text"]].sample(8, random_state=SEED)

In [ ]:
# Group = one line of one agent (all takes of that line). Non-speech clips are grouped by their raw description.
line = np.where(data["has_speech"], data["spoken_text"], data["transcript"]).astype(str)
data["line_group"] = data["agent"] + "::" + pd.Series(line).str.lower().str.strip()

rng = np.random.default_rng(SEED)
group_split = {}
for agent, groups in data.groupby("agent")["line_group"]:
    groups = sorted(groups.unique())
    rng.shuffle(groups)
    n_test = round(len(groups) * SPLIT_FRACTIONS["test"])
    n_val = round(len(groups) * SPLIT_FRACTIONS["validation"])
    for i, group in enumerate(groups):
        group_split[group] = "test" if i < n_test else "validation" if i < n_test + n_val else "train"

data["split"] = data["line_group"].map(group_split)

# Leakage check: every line group belongs to exactly one split
assert data.groupby("line_group")["split"].nunique().max() == 1

summary = data.groupby("split").agg(
    clips=("audio_name", "size"),
    agents=("agent", "nunique"),
    hours=("duration_s", lambda d: round(d.sum() / 3600, 2)),
    english_only=("english_only", "sum"),
).loc[["train", "validation", "test"]]
summary

In [ ]:
# Save the split manifest (used by every later notebook) and the cleaning record
manifest_cols = ["agent", "audio_name", "split", "duration_s", "transcript", "spoken_text", "has_speech", "english_only"]
data[manifest_cols].sort_values(["agent", "audio_name"]).to_csv(f"{RESULTS_DIR}/splits.csv", index=False)

with open(f"{RESULTS_DIR}/data_cleaning.json", "w") as f:
    json.dump({
        "dataset": DATASET,
        "revision": REVISION,
        "agents": AGENTS,
        "min_duration_s": MIN_DURATION_S,
        "cleaning_steps": cleaning_log,
        "split_sizes": summary.reset_index().to_dict(orient="records"),
        "clips_per_agent_per_split": data.groupby(["agent", "split"]).size().unstack().to_dict(orient="index"),
    }, f, indent=2)

print(os.listdir(RESULTS_DIR))

# Step 3: Choose the 3 candidate pre-trained models

| Key | Hugging Face model | Size | Why it's a candidate |
|---|---|---|---|
| `distilhubert` | `ntu-spml/distilhubert` | ~24M | Compressed (distilled) HuBERT: the smallest and fastest, best suited to the CPU-only Streamlit Cloud |
| `wav2vec2` | `facebook/wav2vec2-base` | ~95M | The standard self-supervised speech model; the most widely used baseline |
| `wavlm` | `microsoft/wavlm-base-plus` | ~95M | Pre-trained with overlapping speakers added, which makes it strong at *speaker* tasks such as identifying who is speaking |

**Why these three:** all are self-supervised speech encoders that take the **same input** (raw 16 kHz audio), so the comparison is fair and the code is shared. They cover a **size/speed trade-off** (24M vs 95M parameters) and include one model built for speaker identity.
*Not chosen:* the Audio Spectrogram Transformer (expects 10 s inputs, so ~1 s clips would be mostly padding, and it is heavier) and Whisper encoders (larger, designed for transcription rather than speaker identity).

**Audio preparation**
- **Resample 48 kHz → 16 kHz:** all three models were pre-trained on 16 kHz speech; other rates degrade their features.
- **Keep at most the first 4 s:** 91.8% of clips are ≤ 4 s and are kept whole. The voice is identifiable within the first seconds, and compute grows with length, so longer clips are trimmed rather than padded.
- **Mono, as stored:** every clip is already single-channel.

In [ ]:
import librosa
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

CANDIDATES = {
    "distilhubert": "ntu-spml/distilhubert",
    "wav2vec2": "facebook/wav2vec2-base",
    "wavlm": "microsoft/wavlm-base-plus",
}

TARGET_SR = 16_000     # sampling rate the models were pre-trained on
MAX_SECONDS = 4.0      # longer clips keep their first 4 s

label2id = {agent: i for i, agent in enumerate(AGENTS)}
id2label = {i: agent for agent, i in label2id.items()}


def load_waveform(wav_bytes):
    """WAV bytes -> mono float32 waveform at 16 kHz, at most MAX_SECONDS long."""
    waveform, sr = sf.read(io.BytesIO(wav_bytes), dtype="float32")
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)
    waveform = waveform[: int(MAX_SECONDS * sr)]
    return librosa.resample(waveform, orig_sr=sr, target_sr=TARGET_SR)


# Decode every clip once (about 1–2 minutes), then free the raw bytes
start = time.perf_counter()
data["waveform"] = data["audio_bytes"].map(load_waveform)
data = data.drop(columns="audio_bytes")
print(f"Decoded {len(data)} clips in {time.perf_counter() - start:.0f}s")


def load_candidate(model_id):
    """Pre-trained speech encoder + a new 26-agent classification head."""
    set_seed(SEED)   # the new head is randomly initialised; seeding makes it reproducible
    feature_extractor = AutoFeatureExtractor.from_pretrained(model_id)
    model = AutoModelForAudioClassification.from_pretrained(
        model_id, num_labels=len(AGENTS), label2id=label2id, id2label=id2label,
    )
    # The convolutional feature encoder (low-level acoustic features) is already general;
    # freezing it is the standard recipe and reduces compute and overfitting.
    model.freeze_feature_encoder()
    return feature_extractor, model


for key, model_id in CANDIDATES.items():
    _, m = load_candidate(model_id)
    print(f"{key:<13} {model_id:<28} {m.num_parameters() / 1e6:6.1f}M parameters")
    del m

# Step 4: Fine-tune and evaluate each model

All three models get **identical** settings:

| Setting | Value | Why |
|---|---|---|
| Epochs | 5 | Enough for a 9k-clip dataset; the best epoch is kept, so extra epochs cannot hurt the final model |
| Learning rate | 3e-5 (10% warm-up) | Standard value for fine-tuning these speech encoders |
| Batch size | 16 | Fits the T4 GPU for all three models |
| Mixed precision (fp16) | On | About 2× faster on the T4, with no meaningful accuracy cost |
| Best epoch | by **validation accuracy** | Accuracy is the course's main metric, and the 26 classes are near-balanced (296–415 training clips each) |

**Metrics:** accuracy (share of clips assigned to the right agent) and macro-F1 (F1 averaged over agents), on validation (for selection) and on test (final score).

In [ ]:
import shutil

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from transformers import Trainer, TrainingArguments

NUM_EPOCHS = 5
LEARNING_RATE = 3e-5
BATCH_SIZE = 16


class VoiceLines(torch.utils.data.Dataset):
    """One item = {'audio': 16 kHz waveform, 'label': agent id}."""
    def __init__(self, df):
        self.audio = df["waveform"].tolist()
        self.labels = df["agent"].map(label2id).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return {"audio": self.audio[i], "label": self.labels[i]}


class AudioCollator:
    """Pads a batch of waveforms to the longest one and converts them to model inputs."""
    def __init__(self, feature_extractor):
        self.feature_extractor = feature_extractor

    def __call__(self, batch):
        inputs = self.feature_extractor(
            [item["audio"] for item in batch], sampling_rate=TARGET_SR, padding=True, return_tensors="pt",
        )
        inputs["labels"] = torch.tensor([item["label"] for item in batch])
        return inputs


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
    }


splits = {name: data[data["split"] == name].reset_index(drop=True) for name in ["train", "validation", "test"]}
datasets = {name: VoiceLines(df) for name, df in splits.items()}
print({name: len(ds) for name, ds in datasets.items()})

In [ ]:
results = {}            # metrics per model (becomes the comparison table)
test_predictions = splits["test"][["agent", "audio_name"]].copy()

for key, model_id in CANDIDATES.items():
    print(f"\n{'=' * 20} {key}: {model_id} {'=' * 20}")
    feature_extractor, model = load_candidate(model_id)

    args = TrainingArguments(
        output_dir=f"checkpoints/{key}",
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=0.1,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=32,
        fp16=torch.cuda.is_available(),
        eval_strategy="epoch",              # score validation after each epoch...
        save_strategy="epoch",
        load_best_model_at_end=True,        # ...and keep the best epoch
        metric_for_best_model="accuracy",
        save_total_limit=1,
        logging_strategy="epoch",
        remove_unused_columns=False,        # our items are {'audio', 'label'}, prepared by the collator
        seed=SEED,
        data_seed=SEED,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=datasets["train"],
        eval_dataset=datasets["validation"],
        data_collator=AudioCollator(feature_extractor),
        processing_class=feature_extractor,   # saved with the model, so pipeline() can load it later
        compute_metrics=compute_metrics,
    )

    train_output = trainer.train()
    val_metrics = trainer.evaluate()                       # validation: used for model selection
    test_output = trainer.predict(datasets["test"])        # test: final score
    test_pred = [id2label[i] for i in np.argmax(test_output.predictions, axis=-1)]
    test_predictions[f"pred_{key}"] = test_pred

    best_epoch = max(
        (log for log in trainer.state.log_history if "eval_accuracy" in log),
        key=lambda log: log["eval_accuracy"],
    )["epoch"]

    results[key] = {
        "model_id": model_id,
        "params_millions": round(model.num_parameters() / 1e6, 1),
        "best_epoch": best_epoch,
        "val_accuracy": val_metrics["eval_accuracy"],
        "val_f1_macro": val_metrics["eval_f1_macro"],
        "test_accuracy": test_output.metrics["test_accuracy"],
        "test_f1_macro": test_output.metrics["test_f1_macro"],
        "train_runtime_s": train_output.metrics["train_runtime"],
        "val_eval_runtime_s": val_metrics["eval_runtime"],
    }

    # Detailed per-model record: per-agent scores, confusion matrix, training curve
    with open(f"{RESULTS_DIR}/{key}_details.json", "w") as f:
        json.dump({
            "model_id": model_id,
            "test_classification_report": classification_report(
                splits["test"]["agent"], test_pred, labels=AGENTS, output_dict=True, zero_division=0),
            "test_confusion_matrix": {
                "labels": AGENTS,   # rows = true agent, columns = predicted agent
                "matrix": confusion_matrix(splits["test"]["agent"], test_pred, labels=AGENTS).tolist(),
            },
            "log_history": trainer.state.log_history,
        }, f, indent=2)

    print(f"test accuracy {results[key]['test_accuracy']:.3f} | test macro-F1 {results[key]['test_f1_macro']:.3f}")

    # Keep the best model for the next steps; drop the checkpoints and free GPU memory
    trainer.save_model(f"models/{key}")
    shutil.rmtree(f"checkpoints/{key}", ignore_errors=True)
    del trainer, model
    torch.cuda.empty_cache()

test_predictions.to_csv(f"{RESULTS_DIR}/test_predictions.csv", index=False)
pd.DataFrame(results).T

# Save and download the results

Every file this notebook generated is listed with its **SHA-256 fingerprint** in `SHA256SUMS`.
Rerunning the notebook must reproduce the same fingerprints for the data files (split and cleaning record).
Training files may differ slightly between runs: GPU computation is not bit-for-bit deterministic, and they include runtimes.
Then everything is zipped and downloaded as `results_agent_classifier.zip`.

In [ ]:
import hashlib
from pathlib import Path

result_files = sorted(p for p in Path(RESULTS_DIR).rglob("*") if p.is_file() and p.name != "SHA256SUMS")
with open(f"{RESULTS_DIR}/SHA256SUMS", "w") as f:
    for p in result_files:
        digest = hashlib.sha256(p.read_bytes()).hexdigest()
        f.write(f"{digest}  {p.relative_to(RESULTS_DIR)}\n")
        print(f"{digest[:16]}…  {p.relative_to(RESULTS_DIR)}  ({p.stat().st_size:,} bytes)")

In [ ]:
!rm -f results_agent_classifier.zip && zip -qrX results_agent_classifier.zip results
from google.colab import files
files.download("results_agent_classifier.zip")